# Predictive Maintenance & OEE Command Center

**Snowflake Hackathon — Complete Build Walkthrough**

This notebook contains every SQL step to reproduce the full system from scratch.

## Architecture
```
RAW → BRONZE → SILVER → GOLD
       4 DTs      3 DTs     1 DT     (8 Dynamic Tables, 15-min lag)
       cleanse    features   OEE      (all incremental)
       dedup      24h stats
                  PIT joins
```

**Snowflake Services Used:** Dynamic Tables (8), ML ANOMALY_DETECTION (2), Cortex Agent, Cortex Search, Semantic View, DATA_AGENT_RUN, Streamlit in Snowflake, SYSTEM$SEND_EMAIL, Tasks (5), Stored Procedures (7)

**Validated Results:** 100% detection recall, 84.6% actionable recall, 73.3% precision, 0.85 confidence guardrail

---
## Phase 1: Database, Schemas & RAW Tables

Create the medallion architecture foundation: 4 schemas (RAW, BRONZE, SILVER, GOLD) and all base tables.

- **SENSOR_STREAM**: 768K sensor readings (vibration, temperature, RPM) across 50 assets
- **ASSET_MASTER**: 50 assets across 3 wind farms, 5 subsystem types
- **ERP_WORKORDER**: Historical and agent-created work orders
- **MAINTENANCE_LOG**: Technician notes for Cortex Search
- **PRODUCTION_LOG**: Shift-level OEE source data
- **REPLAY_BUFFER**: Hourly batch drip for simulated streaming
- **AGENT_ACTION_LOG**: Full audit trail of agent decisions

In [ ]:
CREATE DATABASE IF NOT EXISTS PDM_OEE_DB;
CREATE SCHEMA IF NOT EXISTS PDM_OEE_DB.RAW;
CREATE SCHEMA IF NOT EXISTS PDM_OEE_DB.BRONZE;
CREATE SCHEMA IF NOT EXISTS PDM_OEE_DB.SILVER;
CREATE SCHEMA IF NOT EXISTS PDM_OEE_DB.GOLD;

In [ ]:
-- Sensor telemetry stream (768K readings, 50 assets, 10+1 days)
CREATE OR REPLACE TABLE PDM_OEE_DB.RAW.SENSOR_STREAM (
    READING_ID VARCHAR(40) NOT NULL,
    ASSET_ID VARCHAR(12) NOT NULL,
    TS TIMESTAMP_NTZ(9) NOT NULL,
    VIBRATION_MM_S FLOAT,
    TEMP_C FLOAT,
    RPM FLOAT
);

-- Asset master (50 assets across 3 wind farms)
CREATE OR REPLACE TABLE PDM_OEE_DB.RAW.ASSET_MASTER (
    ASSET_ID VARCHAR(12) NOT NULL,
    WIND_FARM VARCHAR(30),
    TURBINE_ID VARCHAR(10),
    SUBSYSTEM VARCHAR(20),
    ASSET_CRITICALITY VARCHAR(10),
    SITE_PRIORITY NUMBER(38,0)
);

-- ERP work orders
CREATE OR REPLACE TABLE PDM_OEE_DB.RAW.ERP_WORKORDER (
    WO_ID VARCHAR(20) NOT NULL, ASSET_ID VARCHAR(12), TYPE VARCHAR(20),
    OPENED_TS TIMESTAMP_NTZ(9), CLOSED_TS TIMESTAMP_NTZ(9), COST NUMBER(38,0),
    TECHNICIAN VARCHAR(50), FAILURE_CODE VARCHAR(200)
);

-- Maintenance log
CREATE OR REPLACE TABLE PDM_OEE_DB.RAW.MAINTENANCE_LOG (
    LOG_ID VARCHAR(40) NOT NULL, ASSET_ID VARCHAR(12), TS TIMESTAMP_NTZ(9),
    NOTE_TEXT VARCHAR(5000), TECHNICIAN VARCHAR(50)
);

-- Production log (OEE source)
CREATE OR REPLACE TABLE PDM_OEE_DB.RAW.PRODUCTION_LOG (
    PRODUCTION_ID VARCHAR(40) NOT NULL, ASSET_ID VARCHAR(12), SHIFT_DATE DATE,
    SHIFT VARCHAR(10), PLANNED_TIME NUMBER(38,0), RUN_TIME NUMBER(38,0),
    IDEAL_CYCLE_TIME FLOAT, GOOD_UNITS NUMBER(38,0), TOTAL_UNITS NUMBER(38,0)
);

-- Replay buffer (hourly batch drip for Day 11)
CREATE OR REPLACE TABLE PDM_OEE_DB.RAW.REPLAY_BUFFER (
    READING_ID VARCHAR(40) NOT NULL, ASSET_ID VARCHAR(12), TS TIMESTAMP_NTZ(9),
    VIBRATION_MM_S FLOAT, TEMP_C FLOAT, RPM FLOAT, REPLAY_HOUR NUMBER(38,0)
);

-- Agent action log (audit trail)
CREATE OR REPLACE TABLE PDM_OEE_DB.RAW.AGENT_ACTION_LOG (
    ACTION_ID VARCHAR(36) DEFAULT UUID_STRING(), ACTION_TYPE VARCHAR(30),
    ASSET_ID VARCHAR(12), CONFIDENCE FLOAT, DETAILS VARCHAR(1000),
    CREATED_BY VARCHAR(50) DEFAULT CURRENT_USER(),
    CREATED_AT TIMESTAMP_LTZ(9) DEFAULT CURRENT_TIMESTAMP()
);

-- Notification config
CREATE OR REPLACE TABLE PDM_OEE_DB.GOLD.NOTIFICATION_CONFIG (
    CONFIG_ID NUMBER(38,0) NOT NULL, REPORT_TYPE VARCHAR(30) NOT NULL,
    RECIPIENT VARCHAR(100) NOT NULL, THRESHOLD FLOAT DEFAULT 0.85
);

### Synthetic Data Generation

Two Python stored procedures generate all sensor, work order, maintenance, and production data:

- **GENERATE_SENSOR_DATA**: 768K readings (10 days x 50 assets). Injects 18 failure events with quadratic degradation curves, including 13 scoring-window failures, 3 training-only failures, and 2 false positives (tent function).
- **GENERATE_REPLAY_BUFFER**: 24K Day 11 rows partitioned by hour. WF02-T03-GBX gets progressive oil contamination degradation.

In [ ]:
-- GENERATE_SENSOR_DATA: 768K sensor readings with 18 injected failure events
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.RAW.GENERATE_SENSOR_DATA()
RETURNS VARCHAR LANGUAGE PYTHON RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python') HANDLER = 'run' EXECUTE AS CALLER
AS $$
import random, math
from datetime import datetime, timedelta

def run(session):
    random.seed(42)
    BASELINES = {
        'GBX': (3.5, 0.5, 55, 3, 1750, 15), 'MBR': (2.8, 0.4, 50, 2.5, 1750, 15),
        'GEN': (2.0, 0.3, 65, 3, 1800, 10), 'PIT': (1.5, 0.3, 40, 2, 120, 5),
        'YAW': (1.8, 0.3, 45, 2, 8, 1),
    }
    FAILURES = [
        ('WF01-T01-GEN', 6, 3, 2.5, 1.4, False), ('WF01-T01-PIT', 7, 3, 2.0, 1.3, False),
        ('WF01-T03-PIT', 6, 3, 2.2, 1.3, False), ('WF02-T01-YAW', 7, 3, 2.8, 1.5, False),
        ('WF02-T02-PIT', 7, 3, 2.3, 1.4, False), ('WF02-T03-MBR', 6, 3, 2.6, 1.5, False),
        ('WF03-T01-YAW', 7, 3, 2.4, 1.4, False), ('WF03-T02-GEN', 6, 3, 2.3, 1.3, False),
        ('WF03-T02-YAW', 6, 3, 2.7, 1.5, False), ('WF03-T03-GBX', 6, 3, 2.5, 1.4, False),
        ('WF01-T02-GBX', 7, 3, 2.4, 1.4, False), ('WF01-T04-GBX', 7, 3, 2.6, 1.5, False),
        ('WF03-T01-MBR', 7, 3, 2.3, 1.3, False),
        ('WF01-T03-MBR', 3, 3, 2.5, 1.4, False), ('WF02-T01-GBX', 4, 3, 2.4, 1.4, False),
        ('WF02-T02-GEN', 4, 3, 2.2, 1.3, False),
        ('WF01-T04-MBR', 5, 2, 1.5, 1.15, True), ('WF02-T03-GEN', 5, 2, 1.6, 1.2, True),
    ]
    failure_map = {}
    for (aid, start_day, dur, vm, tm, is_fp) in FAILURES:
        failure_map[aid] = ((start_day-1)*1440, (start_day-1)*1440 + dur*1440, vm, tm, is_fp)

    assets = session.sql("SELECT ASSET_ID, SUBSYSTEM_CODE FROM PDM_OEE_DB.RAW.ASSET_MASTER ORDER BY ASSET_ID").collect()
    TOTAL_MINUTES = 10 * 1440
    BASE_TS = datetime(2026, 1, 1)
    reading_id = 0; batch = []; total_inserted = 0

    for row in assets:
        aid, code = row['ASSET_ID'], row['SUBSYSTEM_CODE']
        vib_mu, vib_sig, temp_mu, temp_sig, rpm_mu, rpm_sig = BASELINES[code]
        has_failure = aid in failure_map
        if has_failure: f_start, f_end, vib_mult, temp_mult, is_fp = failure_map[aid]

        for minute in range(TOTAL_MINUTES):
            reading_id += 1
            ts = BASE_TS + timedelta(minutes=minute)
            vib = max(0.1, random.gauss(vib_mu, vib_sig))
            temp = max(10, random.gauss(temp_mu, temp_sig))
            rpm = max(0, random.gauss(rpm_mu, rpm_sig))

            if has_failure and f_start <= minute < f_end:
                progress = (minute - f_start) / (f_end - f_start)
                if is_fp:
                    scale = progress * 2 if progress < 0.5 else (1 - progress) * 2
                    vib = max(0.1, random.gauss(vib_mu * (1 + (vib_mult-1)*scale), vib_sig*1.2))
                    temp = max(10, random.gauss(temp_mu * (1 + (temp_mult-1)*scale), temp_sig*1.2))
                else:
                    vib = max(0.1, random.gauss(vib_mu * (1 + (vib_mult-1)*progress**2), vib_sig*(1+progress)))
                    temp = max(10, random.gauss(temp_mu * (1 + (temp_mult-1)*progress**2), temp_sig*(1+progress)))

            batch.append((reading_id, aid, ts.strftime('%Y-%m-%d %H:%M:%S'), round(vib,4), round(temp,2), round(rpm,2)))
            if len(batch) >= 50000:
                df = session.create_dataframe(batch, schema=['READING_ID','ASSET_ID','TS','VIBRATION_MM_S','TEMP_C','RPM'])
                df.write.mode('append').save_as_table('PDM_OEE_DB.RAW.SENSOR_STREAM')
                total_inserted += len(batch); batch = []

    if batch:
        df = session.create_dataframe(batch, schema=['READING_ID','ASSET_ID','TS','VIBRATION_MM_S','TEMP_C','RPM'])
        df.write.mode('append').save_as_table('PDM_OEE_DB.RAW.SENSOR_STREAM')
        total_inserted += len(batch)
    return f'Generated {total_inserted} sensor readings for {len(assets)} assets'
$$;

In [ ]:
-- GENERATE_REPLAY_BUFFER: Day 11 data with WF02-T03-GBX degradation
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.RAW.GENERATE_REPLAY_BUFFER()
RETURNS VARCHAR LANGUAGE PYTHON RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python') HANDLER = 'run' EXECUTE AS CALLER
AS $$
import random
from datetime import datetime, timedelta

def run(session):
    random.seed(99)
    BASELINES = {
        'GBX': (3.5, 0.5, 55, 3, 1750, 15), 'MBR': (2.8, 0.4, 50, 2.5, 1750, 15),
        'GEN': (2.0, 0.3, 65, 3, 1800, 10), 'PIT': (1.5, 0.3, 40, 2, 120, 5),
        'YAW': (1.8, 0.3, 45, 2, 8, 1),
    }
    DEGRADE_ASSET = 'WF02-T03-GBX'
    VIB_MULT, TEMP_MULT = 2.8, 1.5

    assets = session.sql("SELECT ASSET_ID, SUBSYSTEM_CODE FROM PDM_OEE_DB.RAW.ASSET_MASTER ORDER BY ASSET_ID").collect()
    BASE_TS = datetime(2026, 1, 11)
    max_rid = session.sql("SELECT COALESCE(MAX(READING_ID), 0) FROM PDM_OEE_DB.RAW.SENSOR_STREAM").collect()[0][0]
    reading_id = max_rid; batch = []; total = 0

    for row in assets:
        aid, code = row['ASSET_ID'], row['SUBSYSTEM_CODE']
        vib_mu, vib_sig, temp_mu, temp_sig, rpm_mu, rpm_sig = BASELINES[code]

        for minute in range(1440):  # 24 hours
            reading_id += 1
            ts = BASE_TS + timedelta(minutes=minute)
            hour = minute / 60.0
            vib = max(0.1, random.gauss(vib_mu, vib_sig))
            temp = max(10, random.gauss(temp_mu, temp_sig))
            rpm = max(0, random.gauss(rpm_mu, rpm_sig))

            if aid == DEGRADE_ASSET:
                progress = min(1.0, hour / 24.0)
                vib = max(0.1, random.gauss(vib_mu * (1 + (VIB_MULT-1)*progress**2), vib_sig*(1+progress)))
                temp = max(10, random.gauss(temp_mu * (1 + (TEMP_MULT-1)*progress**2), temp_sig*(1+progress)))

            batch.append((reading_id, aid, ts.strftime('%Y-%m-%d %H:%M:%S'), round(vib,4), round(temp,2), round(rpm,2), int(hour)))
            if len(batch) >= 50000:
                df = session.create_dataframe(batch, schema=['READING_ID','ASSET_ID','TS','VIBRATION_MM_S','TEMP_C','RPM','REPLAY_HOUR'])
                df.write.mode('append').save_as_table('PDM_OEE_DB.RAW.REPLAY_BUFFER')
                total += len(batch); batch = []

    if batch:
        df = session.create_dataframe(batch, schema=['READING_ID','ASSET_ID','TS','VIBRATION_MM_S','TEMP_C','RPM','REPLAY_HOUR'])
        df.write.mode('append').save_as_table('PDM_OEE_DB.RAW.REPLAY_BUFFER')
        total += len(batch)
    return f'Generated {total} replay rows for {len(assets)} assets (Day 11, 24h). WF02-T03-GBX degrading.'
$$;

### Run Data Generation
Execute these after creating the procedures above. GENERATE_SENSOR_DATA takes ~2 minutes (768K rows). GENERATE_REPLAY_BUFFER takes ~30 seconds (24K rows).

In [ ]:
CALL PDM_OEE_DB.RAW.GENERATE_SENSOR_DATA();
CALL PDM_OEE_DB.RAW.GENERATE_REPLAY_BUFFER();

> **Note:** After creating tables, load synthetic data into RAW tables. The synthetic data generator populates 768K sensor readings across 50 assets (Days 1-10 normal, Day 11 with injected anomalies for 13 ground-truth assets), 50 assets across 3 wind farms x 5 subsystems, historical work orders, maintenance notes, and production shift records.

---
## Phase 2: Medallion Pipeline — 8 Dynamic Tables

### BRONZE Layer (4 DTs)
Cleanse, deduplicate, standardize. Each table uses `QUALIFY ROW_NUMBER()` to handle exact duplicates.

### SILVER Layer (3 DTs)
- **SILVER_SENSOR_ENRICHED**: Feature engineering with 1-hour rolling stats + point-in-time work order joins (prevents data leakage)
- **SILVER_SENSOR_STATS_24H**: 15-minute bucket aggregates with 24-hour sliding windows
- **SILVER_WORKORDER_ENRICHED**: Work orders joined with maintenance notes by time window

### GOLD Layer (1 DT)
- **GOLD_OEE_METRICS**: OEE = Availability x Performance x Quality per asset per shift

All DTs use `TARGET_LAG = '15 minutes'` and `REFRESH_MODE = AUTO` (Snowflake chooses incremental where possible).

In [ ]:
-- BRONZE: Cleansed sensor stream
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.BRONZE.BRONZE_SENSOR_STREAM
    TARGET_LAG = '15 minutes' REFRESH_MODE = AUTO INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
SELECT READING_ID, ASSET_ID, TS,
    ROUND(VIBRATION_MM_S, 4) AS VIBRATION_MM_S, ROUND(TEMP_C, 2) AS TEMP_C, ROUND(RPM, 2) AS RPM
FROM PDM_OEE_DB.RAW.SENSOR_STREAM
QUALIFY ROW_NUMBER() OVER (PARTITION BY ASSET_ID, TS ORDER BY READING_ID) = 1;

-- BRONZE: Cleansed work orders
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.BRONZE.BRONZE_ERP_WORKORDER
    TARGET_LAG = '15 minutes' REFRESH_MODE = AUTO INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
SELECT WO_ID, ASSET_ID, UPPER(TRIM(TYPE)) AS WO_TYPE, OPENED_TS, CLOSED_TS, COST,
    TRIM(TECHNICIAN) AS TECHNICIAN, UPPER(TRIM(FAILURE_CODE)) AS FAILURE_CODE,
    CASE WHEN CLOSED_TS IS NOT NULL THEN 'COMPLETED' ELSE 'OPEN' END AS STATUS,
    CASE WHEN CLOSED_TS IS NOT NULL THEN DATEDIFF('hour', OPENED_TS, CLOSED_TS) END AS DURATION_HOURS
FROM PDM_OEE_DB.RAW.ERP_WORKORDER
QUALIFY ROW_NUMBER() OVER (PARTITION BY WO_ID ORDER BY OPENED_TS) = 1;

-- BRONZE: Cleansed maintenance log
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.BRONZE.BRONZE_MAINTENANCE_LOG
    TARGET_LAG = '15 minutes' REFRESH_MODE = AUTO INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
SELECT LOG_ID, ASSET_ID, TS, TRIM(NOTE_TEXT) AS NOTE_TEXT, TRIM(TECHNICIAN) AS TECHNICIAN,
    LENGTH(NOTE_TEXT) AS NOTE_LENGTH
FROM PDM_OEE_DB.RAW.MAINTENANCE_LOG
QUALIFY ROW_NUMBER() OVER (PARTITION BY ASSET_ID, TS, TECHNICIAN ORDER BY LOG_ID) = 1;

-- BRONZE: Cleansed production log
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.BRONZE.BRONZE_PRODUCTION_LOG
    TARGET_LAG = '15 minutes' REFRESH_MODE = AUTO INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
SELECT PRODUCTION_ID, ASSET_ID, SHIFT_DATE, UPPER(TRIM(SHIFT)) AS SHIFT,
    PLANNED_TIME, GREATEST(0, LEAST(RUN_TIME, PLANNED_TIME)) AS RUN_TIME,
    IDEAL_CYCLE_TIME, GOOD_UNITS, TOTAL_UNITS
FROM PDM_OEE_DB.RAW.PRODUCTION_LOG
WHERE PLANNED_TIME > 0 AND TOTAL_UNITS >= 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY ASSET_ID, SHIFT_DATE, SHIFT ORDER BY PRODUCTION_ID) = 1;

In [ ]:
-- SILVER: Feature engineering with point-in-time work order joins
-- Prevents data leakage: model never sees future work orders
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.SILVER.SILVER_SENSOR_ENRICHED
    TARGET_LAG = '15 minutes' REFRESH_MODE = INCREMENTAL INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
WITH timeline AS (
    SELECT s.READING_ID, s.ASSET_ID, s.TS, s.VIBRATION_MM_S, s.TEMP_C, s.RPM,
        NULL::TIMESTAMP_NTZ AS WO_CLOSED_TS, 0 AS ROW_ORD
    FROM PDM_OEE_DB.BRONZE.BRONZE_SENSOR_STREAM s
    UNION ALL
    SELECT NULL, w.ASSET_ID, w.CLOSED_TS, NULL, NULL, NULL, w.CLOSED_TS, -1
    FROM PDM_OEE_DB.BRONZE.BRONZE_ERP_WORKORDER w WHERE w.CLOSED_TS IS NOT NULL
),
with_pit AS (
    SELECT *, LAST_VALUE(WO_CLOSED_TS IGNORE NULLS) OVER (
        PARTITION BY ASSET_ID ORDER BY TS, ROW_ORD
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS PIT_LAST_WO_CLOSED FROM timeline
),
sensor_only AS (SELECT * FROM with_pit WHERE ROW_ORD = 0)
SELECT s.READING_ID, s.ASSET_ID, s.TS,
    a.WIND_FARM, a.TURBINE_ID, a.SUBSYSTEM, a.ASSET_CRITICALITY, a.SITE_PRIORITY,
    s.VIBRATION_MM_S, s.TEMP_C, s.RPM,
    AVG(s.VIBRATION_MM_S) OVER (PARTITION BY s.ASSET_ID ORDER BY s.TS ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS VIB_1H_AVG,
    STDDEV(s.VIBRATION_MM_S) OVER (PARTITION BY s.ASSET_ID ORDER BY s.TS ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS VIB_1H_STD,
    AVG(s.TEMP_C) OVER (PARTITION BY s.ASSET_ID ORDER BY s.TS ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS TEMP_1H_AVG,
    STDDEV(s.TEMP_C) OVER (PARTITION BY s.ASSET_ID ORDER BY s.TS ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS TEMP_1H_STD,
    s.RPM - LAG(s.RPM, 1) OVER (PARTITION BY s.ASSET_ID ORDER BY s.TS) AS RPM_DELTA,
    DATEDIFF('hour', s.PIT_LAST_WO_CLOSED, s.TS) AS HOURS_SINCE_LAST_WO
FROM sensor_only s JOIN PDM_OEE_DB.RAW.ASSET_MASTER a ON s.ASSET_ID = a.ASSET_ID;

In [ ]:
-- SILVER: 24-hour aggregated statistics (15-min buckets, 96 per day)
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.SILVER.SILVER_SENSOR_STATS_24H
    TARGET_LAG = '15 minutes' REFRESH_MODE = INCREMENTAL INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
WITH buckets AS (
    SELECT s.ASSET_ID, a.WIND_FARM, a.TURBINE_ID, a.SUBSYSTEM, a.ASSET_CRITICALITY, a.SITE_PRIORITY,
        TIME_SLICE(s.TS, 15, 'MINUTE', 'START') AS BUCKET_TS,
        AVG(s.VIBRATION_MM_S) AS VIB_BUCKET_AVG, STDDEV(s.VIBRATION_MM_S) AS VIB_BUCKET_STD,
        MAX(s.VIBRATION_MM_S) AS VIB_BUCKET_MAX, AVG(s.TEMP_C) AS TEMP_BUCKET_AVG,
        STDDEV(s.TEMP_C) AS TEMP_BUCKET_STD, MAX(s.TEMP_C) AS TEMP_BUCKET_MAX,
        AVG(s.RPM) AS RPM_BUCKET_AVG, COUNT(*) AS READINGS_IN_BUCKET
    FROM PDM_OEE_DB.BRONZE.BRONZE_SENSOR_STREAM s
    JOIN PDM_OEE_DB.RAW.ASSET_MASTER a ON s.ASSET_ID = a.ASSET_ID
    GROUP BY ALL
)
SELECT *, AVG(VIB_BUCKET_AVG) OVER w AS VIB_24H_AVG, STDDEV(VIB_BUCKET_AVG) OVER w AS VIB_24H_STD,
    MAX(VIB_BUCKET_MAX) OVER w AS VIB_24H_MAX, AVG(TEMP_BUCKET_AVG) OVER w AS TEMP_24H_AVG,
    STDDEV(TEMP_BUCKET_AVG) OVER w AS TEMP_24H_STD, MAX(TEMP_BUCKET_MAX) OVER w AS TEMP_24H_MAX,
    VIB_BUCKET_AVG - LAG(VIB_BUCKET_AVG, 24) OVER (PARTITION BY ASSET_ID ORDER BY BUCKET_TS) AS VIB_6H_DELTA,
    TEMP_BUCKET_AVG - LAG(TEMP_BUCKET_AVG, 24) OVER (PARTITION BY ASSET_ID ORDER BY BUCKET_TS) AS TEMP_6H_DELTA
FROM buckets
WINDOW w AS (PARTITION BY ASSET_ID ORDER BY BUCKET_TS ROWS BETWEEN 95 PRECEDING AND CURRENT ROW);

-- SILVER: Work order enrichment
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.SILVER.SILVER_WORKORDER_ENRICHED
    TARGET_LAG = '15 minutes' REFRESH_MODE = AUTO INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
SELECT w.WO_ID, w.ASSET_ID, a.WIND_FARM, a.TURBINE_ID, a.SUBSYSTEM, a.ASSET_CRITICALITY, a.SITE_PRIORITY,
    w.WO_TYPE, w.OPENED_TS, w.CLOSED_TS, w.STATUS, w.DURATION_HOURS, w.COST, w.TECHNICIAN, w.FAILURE_CODE,
    COALESCE(m.NOTE_COUNT, 0) AS NOTE_COUNT, m.COMBINED_NOTES
FROM PDM_OEE_DB.BRONZE.BRONZE_ERP_WORKORDER w
JOIN PDM_OEE_DB.RAW.ASSET_MASTER a ON w.ASSET_ID = a.ASSET_ID
LEFT JOIN (
    SELECT ml.ASSET_ID, wo.WO_ID, COUNT(*) AS NOTE_COUNT,
        LISTAGG(ml.NOTE_TEXT, ' | ') WITHIN GROUP (ORDER BY ml.TS) AS COMBINED_NOTES
    FROM PDM_OEE_DB.BRONZE.BRONZE_MAINTENANCE_LOG ml
    JOIN PDM_OEE_DB.BRONZE.BRONZE_ERP_WORKORDER wo ON ml.ASSET_ID = wo.ASSET_ID
        AND ml.TS BETWEEN wo.OPENED_TS AND COALESCE(wo.CLOSED_TS, DATEADD('day', 365, wo.OPENED_TS))
    GROUP BY ml.ASSET_ID, wo.WO_ID
) m ON w.ASSET_ID = m.ASSET_ID AND w.WO_ID = m.WO_ID;

-- GOLD: OEE = Availability x Performance x Quality
CREATE OR REPLACE DYNAMIC TABLE PDM_OEE_DB.GOLD.GOLD_OEE_METRICS
    TARGET_LAG = '15 minutes' REFRESH_MODE = AUTO INITIALIZE = ON_CREATE WAREHOUSE = COMPUTE_WH
AS
SELECT p.ASSET_ID, a.WIND_FARM, a.TURBINE_ID, a.SUBSYSTEM, a.ASSET_CRITICALITY, a.SITE_PRIORITY,
    p.SHIFT_DATE, p.SHIFT, p.PLANNED_TIME, p.RUN_TIME, p.IDEAL_CYCLE_TIME,
    p.TOTAL_UNITS, p.GOOD_UNITS, p.TOTAL_UNITS - p.GOOD_UNITS AS REJECT_UNITS,
    ROUND(p.RUN_TIME / NULLIF(p.PLANNED_TIME, 0), 4) AS AVAILABILITY,
    ROUND((p.IDEAL_CYCLE_TIME * p.TOTAL_UNITS) / NULLIF(p.RUN_TIME * 60, 0), 4) AS PERFORMANCE,
    ROUND(p.GOOD_UNITS::FLOAT / NULLIF(p.TOTAL_UNITS, 0), 4) AS QUALITY,
    ROUND((p.RUN_TIME / NULLIF(p.PLANNED_TIME, 0)) *
        ((p.IDEAL_CYCLE_TIME * p.TOTAL_UNITS) / NULLIF(p.RUN_TIME * 60, 0)) *
        (p.GOOD_UNITS::FLOAT / NULLIF(p.TOTAL_UNITS, 0)), 4) AS OEE
FROM PDM_OEE_DB.BRONZE.BRONZE_PRODUCTION_LOG p
JOIN PDM_OEE_DB.RAW.ASSET_MASTER a ON p.ASSET_ID = a.ASSET_ID;

---
## Phase 3: ML Anomaly Detection

### Threshold Design (k=13)
- 576 scoring windows per asset (24h x 4 buckets/hr x ~6 feature channels)
- Binomial(n=576, p=0.01): mean=5.76, sigma=2.39
- Threshold: mean + 3*sigma = 5.76 + 7.17 = **13**
- An asset exceeding 13 anomaly flags is outside the 99.87th percentile of normal variation

### Confidence Formula
- `confidence = sqrt((anomaly_count - 13) / 50)`, capped at 1.0
- AUTO_ELIGIBLE >= 0.85, NEEDS_REVIEW >= 0.50, LOGGED_ONLY < 0.50

In [ ]:
-- Scoring views (input to ANOMALY_DETECTION models)
CREATE OR REPLACE VIEW PDM_OEE_DB.GOLD.V_ANOMALY_SCORE_VIB AS
SELECT ASSET_ID AS SERIES, BUCKET_TS AS TS, VIB_BUCKET_AVG AS Y
FROM PDM_OEE_DB.SILVER.SILVER_SENSOR_STATS_24H WHERE BUCKET_TS >= '2026-01-08';

CREATE OR REPLACE VIEW PDM_OEE_DB.GOLD.V_ANOMALY_SCORE_TEMP AS
SELECT ASSET_ID AS SERIES, BUCKET_TS AS TS, TEMP_BUCKET_AVG AS Y
FROM PDM_OEE_DB.SILVER.SILVER_SENSOR_STATS_24H WHERE BUCKET_TS >= '2026-01-08';

-- Training views (Days 1-10 baseline only)
CREATE OR REPLACE VIEW PDM_OEE_DB.GOLD.V_ANOMALY_TRAIN_VIB AS
SELECT ASSET_ID AS SERIES, BUCKET_TS AS TS, VIB_BUCKET_AVG AS Y
FROM PDM_OEE_DB.SILVER.SILVER_SENSOR_STATS_24H WHERE BUCKET_TS < '2026-01-11';

CREATE OR REPLACE VIEW PDM_OEE_DB.GOLD.V_ANOMALY_TRAIN_TEMP AS
SELECT ASSET_ID AS SERIES, BUCKET_TS AS TS, TEMP_BUCKET_AVG AS Y
FROM PDM_OEE_DB.SILVER.SILVER_SENSOR_STATS_24H WHERE BUCKET_TS < '2026-01-11';

In [ ]:
-- Train vibration anomaly model on Days 1-10 baseline
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION PDM_OEE_DB.GOLD.VIB_ANOMALY_MODEL(
    INPUT_DATA => TABLE(PDM_OEE_DB.GOLD.V_ANOMALY_TRAIN_VIB),
    SERIES_COLNAME => 'SERIES', TIMESTAMP_COLNAME => 'TS',
    TARGET_COLNAME => 'Y', LABEL_COLNAME => ''
);

-- Train temperature anomaly model
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION PDM_OEE_DB.GOLD.TEMP_ANOMALY_MODEL(
    INPUT_DATA => TABLE(PDM_OEE_DB.GOLD.V_ANOMALY_TRAIN_TEMP),
    SERIES_COLNAME => 'SERIES', TIMESTAMP_COLNAME => 'TS',
    TARGET_COLNAME => 'Y', LABEL_COLNAME => ''
);

---
## Phase 4: Semantic View & Cortex Search

- **Semantic View** (PDM_OEE_COMMAND_CENTER): Natural language access to OEE, alerts, failures, and asset data. Used by the agent's FleetAnalytics tool.
- **Cortex Search** (MAINTENANCE_SEARCH_SVC): Semantic search over maintenance logs and technician notes. Used by the agent's MaintenanceSearch tool.

In [ ]:
-- Search corpus table
CREATE OR REPLACE TABLE PDM_OEE_DB.GOLD.SEARCH_CORPUS (
    DOC_ID VARCHAR, ASSET_ID VARCHAR(12), SUBSYSTEM VARCHAR(20),
    DOC_TYPE VARCHAR(20), TITLE VARCHAR, SEARCH_TEXT VARCHAR(5000)
);

-- Cortex Search Service
CREATE OR REPLACE CORTEX SEARCH SERVICE PDM_OEE_DB.GOLD.MAINTENANCE_SEARCH_SVC
    ON SEARCH_TEXT
    ATTRIBUTES ASSET_ID, SUBSYSTEM, DOC_TYPE, TITLE
    WAREHOUSE = 'COMPUTE_WH' TARGET_LAG = '1 hour' REFRESH_MODE = INCREMENTAL
AS (SELECT * FROM PDM_OEE_DB.GOLD.SEARCH_CORPUS);

In [ ]:
-- Semantic View: natural language queryable fleet data
CREATE OR REPLACE SEMANTIC VIEW PDM_OEE_DB.GOLD.PDM_OEE_COMMAND_CENTER
    TABLES (
        OEE AS PDM_OEE_DB.GOLD.GOLD_OEE_METRICS PRIMARY KEY (ASSET_ID, SHIFT_DATE, SHIFT),
        FM AS PDM_OEE_DB.GOLD.FLEET_METRICS PRIMARY KEY (WO_ID),
        ALERTS AS PDM_OEE_DB.GOLD.ALERT_QUEUE PRIMARY KEY (ALERT_ID),
        ASSETS AS PDM_OEE_DB.RAW.ASSET_MASTER PRIMARY KEY (ASSET_ID)
    )
    RELATIONSHIPS (
        OEE_TO_ASSETS AS OEE(ASSET_ID) REFERENCES ASSETS(ASSET_ID),
        FM_TO_ASSETS AS FM(ASSET_ID) REFERENCES ASSETS(ASSET_ID),
        ALERTS_TO_ASSETS AS ALERTS(ASSET_ID) REFERENCES ASSETS(ASSET_ID)
    )
    DIMENSIONS (
        OEE.SHIFT_DATE AS SHIFT_DATE COMMENT 'Production shift date',
        OEE.SHIFT AS SHIFT COMMENT 'Shift period' SAMPLE_VALUES ('DAY', 'NIGHT') IS_ENUM,
        FM.FAILURE_CODE_DIM AS FAILURE_CODE COMMENT 'Failure type from ERP',
        FM.SEVERITY_DIM AS SEVERITY COMMENT 'Failure severity' SAMPLE_VALUES ('LOW', 'MEDIUM', 'HIGH') IS_ENUM,
        ALERTS.STATUS_DIM AS STATUS COMMENT 'Alert tier' SAMPLE_VALUES ('AUTO_ELIGIBLE', 'NEEDS_REVIEW', 'LOGGED_ONLY') IS_ENUM,
        ALERTS.DRIVING_SIGNAL_DIM AS DRIVING_SIGNAL COMMENT 'Primary anomaly signal' SAMPLE_VALUES ('VIBRATION', 'TEMPERATURE') IS_ENUM,
        ASSETS.WIND_FARM AS WIND_FARM COMMENT 'Wind farm name',
        ASSETS.TURBINE_ID AS TURBINE_ID COMMENT 'Turbine identifier',
        ASSETS.SUBSYSTEM AS SUBSYSTEM COMMENT 'Subsystem type' SAMPLE_VALUES ('GEARBOX', 'MAIN_BEARING', 'GENERATOR', 'PITCH_SYSTEM', 'YAW_SYSTEM') IS_ENUM,
        ASSETS.ASSET_CRITICALITY AS ASSET_CRITICALITY COMMENT 'Asset criticality' SAMPLE_VALUES ('HIGH', 'MEDIUM', 'LOW') IS_ENUM,
        ASSETS.ASSET_ID_DIM AS ASSET_ID COMMENT 'Unique asset identifier'
    )
    METRICS (
        OEE.AVG_OEE AS AVG(oee.OEE) COMMENT 'Average OEE (0-1)',
        OEE.AVG_AVAILABILITY AS AVG(oee.AVAILABILITY) COMMENT 'Average availability',
        OEE.AVG_PERFORMANCE AS AVG(oee.PERFORMANCE) COMMENT 'Average performance',
        OEE.AVG_QUALITY AS AVG(oee.QUALITY) COMMENT 'Average quality',
        OEE.TOTAL_SHIFTS AS COUNT(oee.SHIFT) COMMENT 'Number of shifts',
        FM.TOTAL_FAILURES AS COUNT(fm.WO_ID) COMMENT 'Number of failures',
        FM.TOTAL_FAILURE_COST AS SUM(fm.FAILURE_COST) COMMENT 'Total failure cost ($)',
        FM.AVG_REPAIR_HOURS AS AVG(fm.REPAIR_HOURS) COMMENT 'Average repair time (MTTR)',
        FM.TOTAL_DOWNTIME AS SUM(fm.FAILURE_DOWNTIME_HOURS) COMMENT 'Total downtime hours',
        ALERTS.TOTAL_ALERTS AS COUNT(alerts.ALERT_ID) COMMENT 'Active alert count',
        ALERTS.AVG_CONFIDENCE AS AVG(alerts.CONFIDENCE) COMMENT 'Average confidence',
        ALERTS.CRITICAL_COUNT AS COUNT_IF(alerts.STATUS = 'AUTO_ELIGIBLE') COMMENT 'Critical alert count',
        ALERTS.AVG_PRIORITY AS AVG(alerts.PRIORITY) COMMENT 'Average alert priority'
    )
    COMMENT = 'Predictive Maintenance & OEE Command Center';

---
## Phase 5: Cortex Agent & Action Procedures

### Agent Tools
1. **FleetAnalytics** — Semantic view (text-to-SQL)
2. **MaintenanceSearch** — Cortex Search (semantic retrieval)
3. **CREATE_WORK_ORDER** — Creates corrective work orders with 0.85 confidence guardrail
4. **NOTIFY_CRITICAL_ALERT** — Sends professional HTML email notifications

### Judge Mode (Human-in-the-Loop)
- **ON:** Agent analyzes -> Confirm/Deny buttons -> human clicks -> procedures execute
- **OFF:** Agent analyzes -> auto-execute when confidence >= 0.85

In [ ]:
-- CREATE_WORK_ORDER: Guardrail at 0.85 confidence
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.GOLD.CREATE_WORK_ORDER(
    P_ASSET_ID VARCHAR, P_CONFIDENCE FLOAT, P_DESCRIPTION VARCHAR, P_OVERRIDE BOOLEAN DEFAULT FALSE
) RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS
BEGIN
    IF (:P_CONFIDENCE < 0.85 AND :P_OVERRIDE = FALSE) THEN
        LET refuse_msg VARCHAR := 'Confidence ' || :P_CONFIDENCE || ' below 0.85 threshold. Override not set.';
        INSERT INTO PDM_OEE_DB.RAW.AGENT_ACTION_LOG (ACTION_TYPE, ASSET_ID, CONFIDENCE, DETAILS)
        VALUES ('WO_REFUSED', :P_ASSET_ID, :P_CONFIDENCE, :refuse_msg);
        RETURN 'REFUSED: ' || :refuse_msg;
    END IF;
    LET wo_id VARCHAR := 'WO-AGT-' || LEFT(REPLACE(UUID_STRING(), '-', ''), 12);
    INSERT INTO PDM_OEE_DB.RAW.ERP_WORKORDER (WO_ID, ASSET_ID, TYPE, OPENED_TS, FAILURE_CODE)
    VALUES (:wo_id, :P_ASSET_ID, 'CORRECTIVE', CURRENT_TIMESTAMP(), :P_DESCRIPTION);
    INSERT INTO PDM_OEE_DB.RAW.AGENT_ACTION_LOG (ACTION_TYPE, ASSET_ID, CONFIDENCE, DETAILS)
    VALUES ('WO_CREATED', :P_ASSET_ID, :P_CONFIDENCE, 'Work order ' || :wo_id || ' created.');
    RETURN 'SUCCESS: Work order ' || :wo_id || ' created for ' || :P_ASSET_ID;
END;

In [ ]:
-- NOTIFY_CRITICAL_ALERT: Professional HTML email via SYSTEM$SEND_EMAIL
-- Gathers asset details, alert info, failure history, and maintenance notes
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.GOLD.NOTIFY_CRITICAL_ALERT(
    P_ASSET_ID VARCHAR, P_CONFIDENCE FLOAT, P_MESSAGE VARCHAR
) RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS
BEGIN
    LET v_subsystem VARCHAR := (SELECT SUBSYSTEM FROM PDM_OEE_DB.RAW.ASSET_MASTER WHERE ASSET_ID = :P_ASSET_ID);
    LET v_wind_farm VARCHAR := (SELECT WIND_FARM FROM PDM_OEE_DB.RAW.ASSET_MASTER WHERE ASSET_ID = :P_ASSET_ID);
    LET v_turbine VARCHAR := (SELECT TURBINE_ID FROM PDM_OEE_DB.RAW.ASSET_MASTER WHERE ASSET_ID = :P_ASSET_ID);
    LET v_criticality VARCHAR := (SELECT ASSET_CRITICALITY FROM PDM_OEE_DB.RAW.ASSET_MASTER WHERE ASSET_ID = :P_ASSET_ID);
    LET v_priority INT := (SELECT PRIORITY FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE ASSET_ID = :P_ASSET_ID);
    LET v_signal VARCHAR := (SELECT DRIVING_SIGNAL FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE ASSET_ID = :P_ASSET_ID);
    LET v_status VARCHAR := (SELECT STATUS FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE ASSET_ID = :P_ASSET_ID);
    LET v_wo_id VARCHAR := (SELECT WO_ID FROM PDM_OEE_DB.RAW.ERP_WORKORDER
        WHERE ASSET_ID = :P_ASSET_ID AND WO_ID LIKE 'WO-AGT%' ORDER BY OPENED_TS DESC LIMIT 1);
    LET v_failures INT := (SELECT COALESCE(TOTAL_FAILURES, 0) FROM PDM_OEE_DB.GOLD.COMMAND_CENTER_WIDE WHERE ASSET_ID = :P_ASSET_ID);
    LET v_cost INT := (SELECT COALESCE(TOTAL_FAILURE_COST, 0) FROM PDM_OEE_DB.GOLD.COMMAND_CENTER_WIDE WHERE ASSET_ID = :P_ASSET_ID);
    LET v_mttr VARCHAR := (SELECT COALESCE(ROUND(AVG_REPAIR_HOURS, 1)::VARCHAR, 'N/A') FROM PDM_OEE_DB.GOLD.COMMAND_CENTER_WIDE WHERE ASSET_ID = :P_ASSET_ID);

    -- Maintenance notes: asset-specific first, fleet-fallback second
    LET v_notes_html VARCHAR := '';
    LET v_asset_notes VARCHAR := (
        SELECT LISTAGG('<li>' || LEFT(NOTE_TEXT, 120) || '</li>', '') WITHIN GROUP (ORDER BY TS DESC)
        FROM (SELECT NOTE_TEXT, TS FROM PDM_OEE_DB.BRONZE.BRONZE_MAINTENANCE_LOG WHERE ASSET_ID = :P_ASSET_ID ORDER BY TS DESC LIMIT 2)
    );
    IF (:v_asset_notes IS NOT NULL AND :v_asset_notes != '') THEN
        v_notes_html := '<strong>Maintenance Notes (' || :P_ASSET_ID || '):</strong><ul style="margin:4px 0 0 16px;padding:0;">' || :v_asset_notes || '</ul>';
    ELSE
        LET v_fleet_notes VARCHAR := (
            SELECT LISTAGG('<li>' || LEFT(ml.NOTE_TEXT, 120) || '</li>', '') WITHIN GROUP (ORDER BY ml.TS DESC)
            FROM (SELECT ml.NOTE_TEXT, ml.TS FROM PDM_OEE_DB.BRONZE.BRONZE_MAINTENANCE_LOG ml
                JOIN PDM_OEE_DB.RAW.ASSET_MASTER am ON ml.ASSET_ID = am.ASSET_ID
                WHERE am.SUBSYSTEM = :v_subsystem ORDER BY ml.TS DESC LIMIT 2) ml
        );
        IF (:v_fleet_notes IS NOT NULL AND :v_fleet_notes != '') THEN
            v_notes_html := '<strong>Maintenance Notes</strong> <span style="color:#888;font-size:12px;">(from fleet ' || :v_subsystem || ' history)</span>:<ul style="margin:4px 0 0 16px;padding:0;">' || :v_fleet_notes || '</ul>';
        END IF;
    END IF;

    LET v_pct VARCHAR := ROUND(:P_CONFIDENCE * 100, 1)::VARCHAR;
    LET v_ts VARCHAR := TO_CHAR(CURRENT_TIMESTAMP(), 'YYYY-MM-DD HH24:MI:SS UTC');
    LET v_subject VARCHAR := '[CRITICAL] Turbine Alert - ' || :P_ASSET_ID || ' (Priority ' || COALESCE(:v_priority::VARCHAR, 'N/A') || ')';

    -- Build history section
    LET v_history_html VARCHAR;
    IF (:v_failures > 0) THEN
        v_history_html :=
            '<table style="width:100%;border-collapse:collapse;font-size:14px;">'
         || '<tr style="background:#f8f9fa;"><td style="padding:8px 12px;font-weight:bold;width:40%;border-bottom:1px solid #eee;">Prior Failures</td>'
         || '<td style="padding:8px 12px;border-bottom:1px solid #eee;">' || :v_failures::VARCHAR || '</td></tr>'
         || '<tr><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Avg Repair Time</td>'
         || '<td style="padding:8px 12px;border-bottom:1px solid #eee;">' || :v_mttr || ' hrs</td></tr>'
         || '<tr style="background:#f8f9fa;"><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Total Failure Cost</td>'
         || '<td style="padding:8px 12px;border-bottom:1px solid #eee;">$' || :v_cost::VARCHAR || '</td></tr></table>';
    ELSE
        v_history_html := '<p style="font-size:14px;color:#7f8c8d;">No prior failures recorded.</p>';
    END IF;
    IF (:v_notes_html != '') THEN
        v_history_html := :v_history_html || '<div style="font-size:13px;margin:10px 0 0;color:#555;">' || :v_notes_html || '</div>';
    END IF;

    -- HTML email body
    LET v_body VARCHAR := '<div style="font-family:Segoe UI,Arial,sans-serif;max-width:600px;margin:0 auto;">'
|| '<div style="background:#c0392b;color:#fff;padding:14px 20px;border-radius:6px 6px 0 0;">'
|| '<h2 style="margin:0;font-size:18px;">CRITICAL ANOMALY ALERT</h2></div>'
|| '<div style="border:1px solid #ddd;border-top:none;padding:20px;border-radius:0 0 6px 6px;">'
|| '<table style="width:100%;border-collapse:collapse;font-size:14px;">'
|| '<tr style="background:#f8f9fa;"><td style="padding:8px 12px;font-weight:bold;width:40%;border-bottom:1px solid #eee;">Asset ID</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || :P_ASSET_ID || '</td></tr>'
|| '<tr><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Subsystem</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || COALESCE(:v_subsystem, 'N/A') || '</td></tr>'
|| '<tr style="background:#f8f9fa;"><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Wind Farm / Turbine</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || COALESCE(:v_wind_farm, '?') || ' / ' || COALESCE(:v_turbine, '?') || '</td></tr>'
|| '<tr><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Criticality</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || COALESCE(:v_criticality, 'N/A') || '</td></tr></table>'
|| '<h3 style="margin:18px 0 10px;color:#c0392b;font-size:15px;">Alert Details</h3>'
|| '<table style="width:100%;border-collapse:collapse;font-size:14px;">'
|| '<tr style="background:#f8f9fa;"><td style="padding:8px 12px;font-weight:bold;width:40%;border-bottom:1px solid #eee;">Confidence</td><td style="padding:8px 12px;border-bottom:1px solid #eee;"><strong>' || :v_pct || '%</strong></td></tr>'
|| '<tr><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Driving Signal</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || COALESCE(:v_signal, 'N/A') || '</td></tr>'
|| '<tr style="background:#f8f9fa;"><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Priority</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || COALESCE(:v_priority::VARCHAR, 'N/A') || '</td></tr></table>'
|| '<h3 style="margin:18px 0 10px;color:#2c3e50;font-size:15px;">Work Order</h3>'
|| '<table style="width:100%;border-collapse:collapse;font-size:14px;">'
|| '<tr style="background:#f8f9fa;"><td style="padding:8px 12px;font-weight:bold;width:40%;border-bottom:1px solid #eee;">Work Order ID</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || COALESCE(:v_wo_id, 'Pending') || '</td></tr>'
|| '<tr><td style="padding:8px 12px;font-weight:bold;border-bottom:1px solid #eee;">Created</td><td style="padding:8px 12px;border-bottom:1px solid #eee;">' || :v_ts || '</td></tr></table>'
|| '<h3 style="margin:18px 0 10px;color:#2c3e50;font-size:15px;">Recent History</h3>'
|| :v_history_html
|| '<p style="margin:18px 0 0;padding:10px;background:#fef9e7;border-left:4px solid #f39c12;font-size:12px;color:#7d6608;">'
|| 'Auto-generated by PDM OEE Command Center. Confidence ' || :v_pct || '% exceeds the 85% auto-action threshold.</p>'
|| '</div></div>';

    INSERT INTO PDM_OEE_DB.RAW.AGENT_ACTION_LOG (ACTION_TYPE, ASSET_ID, CONFIDENCE, DETAILS)
    VALUES ('NOTIFICATION_SENT', :P_ASSET_ID, :P_CONFIDENCE, :P_MESSAGE);

    -- Replace with your notification integration and email
    CALL SYSTEM$SEND_EMAIL('PDM_EMAIL_INT', 'recipient@example.com', :v_subject, :v_body, 'text/html');
    RETURN 'SUCCESS: Critical alert notification sent for ' || :P_ASSET_ID;
EXCEPTION
    WHEN OTHER THEN
        INSERT INTO PDM_OEE_DB.RAW.AGENT_ACTION_LOG (ACTION_TYPE, ASSET_ID, CONFIDENCE, DETAILS)
        VALUES ('NOTIFICATION_FAILED', :P_ASSET_ID, :P_CONFIDENCE, 'Email failed: ' || SQLERRM);
        RETURN 'WARNING: Alert logged but email delivery failed: ' || SQLERRM;
END;

In [ ]:
-- SCORING_PROC: Scores sensor data, materializes ALERT_QUEUE
-- Uses Binomial threshold k=13, confidence = sqrt((count-13)/50)
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.GOLD.SCORING_PROC()
RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS
BEGIN
    CREATE OR REPLACE TABLE PDM_OEE_DB.GOLD.VIB_ANOMALY_RESULTS AS
    SELECT * FROM TABLE(PDM_OEE_DB.GOLD.VIB_ANOMALY_MODEL!DETECT_ANOMALIES(
        INPUT_DATA => TABLE(PDM_OEE_DB.GOLD.V_ANOMALY_SCORE_VIB),
        SERIES_COLNAME => 'SERIES', TIMESTAMP_COLNAME => 'TS', TARGET_COLNAME => 'Y',
        CONFIG_OBJECT => {'prediction_interval': 0.99}
    ));

    CREATE OR REPLACE TABLE PDM_OEE_DB.GOLD.TEMP_ANOMALY_RESULTS AS
    SELECT * FROM TABLE(PDM_OEE_DB.GOLD.TEMP_ANOMALY_MODEL!DETECT_ANOMALIES(
        INPUT_DATA => TABLE(PDM_OEE_DB.GOLD.V_ANOMALY_SCORE_TEMP),
        SERIES_COLNAME => 'SERIES', TIMESTAMP_COLNAME => 'TS', TARGET_COLNAME => 'Y',
        CONFIG_OBJECT => {'prediction_interval': 0.99}
    ));

    CREATE OR REPLACE TABLE PDM_OEE_DB.GOLD.ALERT_QUEUE AS
    WITH anomaly_counts AS (
        SELECT v.SERIES AS ASSET_ID, COUNT_IF(v.IS_ANOMALY) AS VIB_ANOM,
            MIN(CASE WHEN v.IS_ANOMALY THEN v.TS END) AS VIB_FIRST_ANOM
        FROM PDM_OEE_DB.GOLD.VIB_ANOMALY_RESULTS v GROUP BY v.SERIES
    ),
    temp_counts AS (
        SELECT t.SERIES AS ASSET_ID, COUNT_IF(t.IS_ANOMALY) AS TEMP_ANOM,
            MIN(CASE WHEN t.IS_ANOMALY THEN t.TS END) AS TEMP_FIRST_ANOM
        FROM PDM_OEE_DB.GOLD.TEMP_ANOMALY_RESULTS t GROUP BY t.SERIES
    ),
    combined AS (
        SELECT a.ASSET_ID, am.ASSET_CRITICALITY, am.SITE_PRIORITY,
            a.VIB_ANOM, t.TEMP_ANOM, a.VIB_ANOM + t.TEMP_ANOM AS TOTAL_ANOMALIES,
            CASE WHEN a.VIB_ANOM >= t.TEMP_ANOM THEN 'VIBRATION' ELSE 'TEMPERATURE' END AS DRIVING_SIGNAL,
            LEAST(COALESCE(a.VIB_FIRST_ANOM, t.TEMP_FIRST_ANOM),
                  COALESCE(t.TEMP_FIRST_ANOM, a.VIB_FIRST_ANOM)) AS FIRST_ANOM_TS
        FROM anomaly_counts a JOIN temp_counts t ON a.ASSET_ID = t.ASSET_ID
        JOIN PDM_OEE_DB.RAW.ASSET_MASTER am ON a.ASSET_ID = am.ASSET_ID
        WHERE a.VIB_ANOM + t.TEMP_ANOM >= 13  -- Binomial threshold k=13
    ),
    with_oee AS (
        SELECT c.*, AVG(o.OEE) AS AVG_OEE FROM combined c
        LEFT JOIN PDM_OEE_DB.GOLD.GOLD_OEE_METRICS o ON c.ASSET_ID = o.ASSET_ID
        GROUP BY c.ASSET_ID, c.ASSET_CRITICALITY, c.SITE_PRIORITY,
                 c.VIB_ANOM, c.TEMP_ANOM, c.TOTAL_ANOMALIES, c.DRIVING_SIGNAL, c.FIRST_ANOM_TS
    )
    SELECT UUID_STRING() AS ALERT_ID, ASSET_ID,
        DATEADD('hour', 48, FIRST_ANOM_TS) AS PREDICTED_FAILURE_TS,
        ROUND(LEAST(1.0, POWER((TOTAL_ANOMALIES - 13.0) / 50.0, 0.5)), 4)::FLOAT AS CONFIDENCE,
        DRIVING_SIGNAL, ROUND(AVG_OEE, 4)::FLOAT AS CURRENT_OEE,
        ROUND((LEAST(1.0, POWER((TOTAL_ANOMALIES - 13.0) / 50.0, 0.5)) * 0.5 +
               CASE ASSET_CRITICALITY WHEN 'HIGH' THEN 1.0 WHEN 'MEDIUM' THEN 0.6 ELSE 0.3 END * 0.3 +
               (1.0 - (SITE_PRIORITY - 1.0) / 2.0) * 0.2) * 100)::INT AS PRIORITY,
        CASE WHEN LEAST(1.0, POWER((TOTAL_ANOMALIES - 13.0) / 50.0, 0.5)) >= 0.85 THEN 'AUTO_ELIGIBLE'
             WHEN LEAST(1.0, POWER((TOTAL_ANOMALIES - 13.0) / 50.0, 0.5)) >= 0.50 THEN 'NEEDS_REVIEW'
             ELSE 'LOGGED_ONLY' END AS STATUS,
        CURRENT_TIMESTAMP() AS CREATED_AT
    FROM with_oee;

    LET alert_count INT := (SELECT COUNT(*) FROM PDM_OEE_DB.GOLD.ALERT_QUEUE);
    LET critical_count INT := (SELECT COUNT(*) FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE STATUS = 'AUTO_ELIGIBLE');
    RETURN 'Scoring complete: ' || :alert_count || ' alerts (' || :critical_count || ' critical)';
END;

In [ ]:
-- CRITICAL_ALERT_PROC: Batch notification for all AUTO_ELIGIBLE alerts
-- Iterates through NOTIFICATION_CONFIG recipients with per-recipient error handling
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.GOLD.CRITICAL_ALERT_PROC()
RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS
DECLARE
    c_recipient CURSOR FOR
        SELECT DISTINCT RECIPIENT FROM PDM_OEE_DB.GOLD.NOTIFICATION_CONFIG
        WHERE REPORT_TYPE = 'CRITICAL_ALERT';
    v_recipient VARCHAR;
    v_success_count INT DEFAULT 0;
    v_fail_count INT DEFAULT 0;
    v_error_msg VARCHAR;
BEGIN
    LET critical_count INT := (SELECT COUNT(*) FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE STATUS = 'AUTO_ELIGIBLE');
    IF (:critical_count = 0) THEN RETURN 'No critical alerts to notify.'; END IF;

    LET summary VARCHAR := (
        SELECT LISTAGG('- ' || ASSET_ID || ' (confidence=' || CONFIDENCE || ', priority=' || PRIORITY || ')', '\n')
        FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE STATUS = 'AUTO_ELIGIBLE' ORDER BY PRIORITY DESC
    );
    LET subject VARCHAR := 'PDM Alert: ' || :critical_count || ' critical anomalies detected';
    LET msg VARCHAR := 'CRITICAL ALERT SUMMARY\n\n' || :critical_count || ' assets require immediate attention:\n\n' || :summary;

    OPEN c_recipient;
    FOR rec IN c_recipient DO
        v_recipient := rec.RECIPIENT;
        BEGIN
            CALL SYSTEM$SEND_EMAIL('PDM_EMAIL_INT', :v_recipient, :subject, :msg);
            INSERT INTO PDM_OEE_DB.RAW.AGENT_ACTION_LOG (ACTION_TYPE, ASSET_ID, CONFIDENCE, DETAILS)
            VALUES ('NOTIFICATION_SENT', 'FLEET', :critical_count, 'Email sent to ' || :v_recipient);
            v_success_count := v_success_count + 1;
        EXCEPTION
            WHEN OTHER THEN
                v_error_msg := 'Email to ' || :v_recipient || ' failed: ' || SQLCODE;
                INSERT INTO PDM_OEE_DB.RAW.AGENT_ACTION_LOG (ACTION_TYPE, ASSET_ID, CONFIDENCE, DETAILS)
                VALUES ('NOTIFICATION_FAILED', 'FLEET', :critical_count, :v_error_msg);
                v_fail_count := v_fail_count + 1;
        END;
    END FOR;
    CLOSE c_recipient;
    RETURN 'Notified ' || :critical_count || ' critical alerts. Sent: ' || :v_success_count || ', Failed: ' || :v_fail_count;
END;

In [ ]:
-- FLEET_SUMMARY_PROC: Daily digest email with fleet OEE + alert counts
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.GOLD.FLEET_SUMMARY_PROC()
RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS
BEGIN
    LET total_assets INT := (SELECT COUNT(*) FROM PDM_OEE_DB.RAW.ASSET_MASTER);
    LET avg_oee FLOAT := (SELECT ROUND(AVG(OEE), 4) FROM PDM_OEE_DB.GOLD.GOLD_OEE_METRICS);
    LET alert_count INT := (SELECT COUNT(*) FROM PDM_OEE_DB.GOLD.ALERT_QUEUE);
    LET critical_count INT := (SELECT COUNT(*) FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE STATUS = 'AUTO_ELIGIBLE');
    LET total_failure_cost INT := (SELECT COALESCE(SUM(FAILURE_COST), 0) FROM PDM_OEE_DB.GOLD.FLEET_METRICS);

    LET msg VARCHAR := 'FLEET DAILY SUMMARY\n\n' ||
        'Total Assets: ' || :total_assets || '\n' ||
        'Fleet Avg OEE: ' || :avg_oee || '\n' ||
        'Active Alerts: ' || :alert_count || ' (' || :critical_count || ' critical)\n' ||
        'Total Failure Cost: $' || :total_failure_cost || '\n';

    BEGIN
        CALL SYSTEM$SEND_EMAIL('PDM_EMAIL_INT', 'recipient@example.com',
            'PDM Fleet Summary: OEE=' || :avg_oee || ', ' || :alert_count || ' alerts', :msg);
    EXCEPTION WHEN OTHER THEN NULL;
    END;
    RETURN 'Fleet summary: ' || :total_assets || ' assets, OEE=' || :avg_oee || ', ' || :alert_count || ' alerts';
END;

-- RETRAIN_CANDIDACY_PROC: Checks if new data volume warrants model retraining
-- Triggers when new data exceeds 5% of baseline (Days 1-10 = 720K rows)
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.GOLD.RETRAIN_CANDIDACY_PROC()
RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS
BEGIN
    LET new_rows INT := (SELECT COUNT(*) FROM PDM_OEE_DB.RAW.SENSOR_STREAM WHERE TS >= '2026-01-11 00:00:00');
    LET baseline_rows INT := (SELECT COUNT(*) FROM PDM_OEE_DB.RAW.SENSOR_STREAM WHERE TS < '2026-01-11 00:00:00');
    LET ratio FLOAT := :new_rows::FLOAT / NULLIF(:baseline_rows, 0);

    IF (:ratio >= 0.05) THEN
        INSERT INTO PDM_OEE_DB.RAW.AGENT_ACTION_LOG (ACTION_TYPE, ASSET_ID, CONFIDENCE, DETAILS)
        VALUES ('RETRAIN_SUGGESTED', 'FLEET', :ratio,
                'New data ratio: ' || ROUND(:ratio * 100, 1) || '% (' || :new_rows || ' new / ' || :baseline_rows || ' baseline). Retrain recommended.');
        BEGIN
            CALL SYSTEM$SEND_EMAIL('PDM_EMAIL_INT', 'recipient@example.com',
                'PDM: Model retrain suggested (' || ROUND(:ratio * 100, 1) || '% new data)',
                'The sensor data has grown by ' || ROUND(:ratio * 100, 1) || '% since baseline. New: ' || :new_rows || '. Baseline: ' || :baseline_rows);
        EXCEPTION WHEN OTHER THEN NULL;
        END;
        RETURN 'RETRAIN_SUGGESTED: ' || :new_rows || ' new rows (' || ROUND(:ratio * 100, 1) || '% of baseline)';
    ELSE
        RETURN 'NO_RETRAIN: ' || :new_rows || ' new rows (' || ROUND(:ratio * 100, 1) || '% of baseline, threshold=5%)';
    END IF;
END;

In [ ]:
-- REPLAY_BATCH_PROC: Drips one hour of Day 11 data per call
CREATE OR REPLACE PROCEDURE PDM_OEE_DB.RAW.REPLAY_BATCH_PROC()
RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS
BEGIN
    LET next_hour INT := (SELECT MIN(REPLAY_HOUR) FROM PDM_OEE_DB.RAW.REPLAY_BUFFER);
    IF (:next_hour IS NULL) THEN RETURN 'REPLAY_COMPLETE: No more data.'; END IF;
    LET batch_rows INT := (SELECT COUNT(*) FROM PDM_OEE_DB.RAW.REPLAY_BUFFER WHERE REPLAY_HOUR = :next_hour);
    INSERT INTO PDM_OEE_DB.RAW.SENSOR_STREAM (READING_ID, ASSET_ID, TS, VIBRATION_MM_S, TEMP_C, RPM)
    SELECT READING_ID, ASSET_ID, TS, VIBRATION_MM_S, TEMP_C, RPM
    FROM PDM_OEE_DB.RAW.REPLAY_BUFFER WHERE REPLAY_HOUR = :next_hour;
    DELETE FROM PDM_OEE_DB.RAW.REPLAY_BUFFER WHERE REPLAY_HOUR = :next_hour;
    LET remaining INT := (SELECT COUNT(DISTINCT REPLAY_HOUR) FROM PDM_OEE_DB.RAW.REPLAY_BUFFER);
    RETURN 'Replayed hour ' || :next_hour || ' (' || :batch_rows || ' rows). ' || :remaining || ' hours remaining.';
END;

In [ ]:
-- Cortex Agent: 4-tool wind turbine maintenance advisor
CREATE OR REPLACE AGENT PDM_OEE_DB.GOLD.PDM_COMMAND_CENTER_AGENT
    COMMENT = 'Wind turbine fleet maintenance advisor'
FROM SPECIFICATION $$
models:
  orchestration: "auto"
instructions:
  response: "You are a wind turbine fleet maintenance advisor. Always check confidence scores before recommending work orders. For confidence >= 0.85, you may auto-create work orders. For 0.50-0.85, recommend but ask for confirmation. Below 0.50, advise monitoring only."
  orchestration: "Use FleetAnalytics for OEE metrics, alert data, and failure statistics. Use MaintenanceSearch for maintenance history. Use CREATE_WORK_ORDER only when confidence >= 0.85 or user explicitly overrides."
tools:
  - tool_spec: { type: "cortex_analyst_text_to_sql", name: "FleetAnalytics", description: "Queries OEE metrics, alert queue, fleet health" }
  - tool_spec: { type: "cortex_search", name: "MaintenanceSearch", description: "Searches maintenance logs and technician notes" }
  - tool_spec:
      type: "generic"
      name: "CREATE_WORK_ORDER"
      description: "Creates a corrective work order when confidence >= 0.85"
      input_schema: { type: "object", properties: { P_ASSET_ID: { type: "string" }, P_CONFIDENCE: { type: "number" }, P_DESCRIPTION: { type: "string" }, P_OVERRIDE: { type: "boolean" } }, required: ["P_ASSET_ID", "P_CONFIDENCE", "P_DESCRIPTION"] }
  - tool_spec:
      type: "generic"
      name: "NOTIFY_CRITICAL_ALERT"
      description: "Sends critical alert email notification"
      input_schema: { type: "object", properties: { P_ASSET_ID: { type: "string" }, P_CONFIDENCE: { type: "number" }, P_MESSAGE: { type: "string" } }, required: ["P_ASSET_ID", "P_CONFIDENCE", "P_MESSAGE"] }
tool_resources:
  FleetAnalytics: { semantic_view: "PDM_OEE_DB.GOLD.PDM_OEE_COMMAND_CENTER", execution_environment: { type: "warehouse", warehouse: "COMPUTE_WH" } }
  MaintenanceSearch: { search_service: "PDM_OEE_DB.GOLD.MAINTENANCE_SEARCH_SVC", max_results: "10" }
  CREATE_WORK_ORDER: { type: "procedure", procedure: "PDM_OEE_DB.GOLD.CREATE_WORK_ORDER", execution_environment: { type: "warehouse", warehouse: "COMPUTE_WH" } }
  NOTIFY_CRITICAL_ALERT: { type: "procedure", procedure: "PDM_OEE_DB.GOLD.NOTIFY_CRITICAL_ALERT", execution_environment: { type: "warehouse", warehouse: "COMPUTE_WH" } }
$$;

---
## Phase 6: Task Automation

| Task | Schedule | Purpose |
|------|----------|---------|
| SCORING_TASK | Every 15 min | Re-score anomalies, refresh ALERT_QUEUE |
| CRITICAL_ALERT_TASK | After scoring | Email critical alerts to configured recipients |
| FLEET_SUMMARY_TASK | Daily 6 AM | Fleet-wide OEE + alert digest |
| RETRAIN_CANDIDACY_TASK | Every 6 hours | Check if new data warrants model retraining |
| REPLAY_TASK | Every 60 min | Drip one hour of Day 11 data into SENSOR_STREAM |

In [ ]:
CREATE OR REPLACE TASK PDM_OEE_DB.GOLD.SCORING_TASK
    WAREHOUSE = COMPUTE_WH SCHEDULE = '15 MINUTE'
    COMMENT = 'Scores sensor data against ML baseline, refreshes ALERT_QUEUE every 15 min'
AS CALL PDM_OEE_DB.GOLD.SCORING_PROC();

CREATE OR REPLACE TASK PDM_OEE_DB.GOLD.CRITICAL_ALERT_TASK
    WAREHOUSE = COMPUTE_WH AFTER PDM_OEE_DB.GOLD.SCORING_TASK
AS CALL PDM_OEE_DB.GOLD.CRITICAL_ALERT_PROC();

CREATE OR REPLACE TASK PDM_OEE_DB.GOLD.FLEET_SUMMARY_TASK
    WAREHOUSE = COMPUTE_WH SCHEDULE = 'USING CRON 0 6 * * * America/Los_Angeles'
AS CALL PDM_OEE_DB.GOLD.FLEET_SUMMARY_PROC();

CREATE OR REPLACE TASK PDM_OEE_DB.GOLD.RETRAIN_CANDIDACY_TASK
    WAREHOUSE = COMPUTE_WH SCHEDULE = 'USING CRON 0 0/6 * * * America/Los_Angeles'
AS CALL PDM_OEE_DB.GOLD.RETRAIN_CANDIDACY_PROC();

CREATE OR REPLACE TASK PDM_OEE_DB.RAW.REPLAY_TASK
    WAREHOUSE = COMPUTE_WH SCHEDULE = '60 MINUTE'
    COMMENT = 'Drips one hour of Day 11 replay data into SENSOR_STREAM per execution'
AS CALL PDM_OEE_DB.RAW.REPLAY_BATCH_PROC();

---
## Verification Queries

Run these to verify the system is working correctly after deployment.

In [ ]:
-- Check Dynamic Table health
SHOW DYNAMIC TABLES IN DATABASE PDM_OEE_DB;

In [ ]:
-- Alert queue breakdown
SELECT STATUS, COUNT(*) AS CNT, ROUND(AVG(CONFIDENCE), 3) AS AVG_CONF
FROM PDM_OEE_DB.GOLD.ALERT_QUEUE GROUP BY STATUS ORDER BY 1;

In [ ]:
-- AUTO_ELIGIBLE assets ready for action
SELECT ASSET_ID, ROUND(CONFIDENCE, 4) AS CONF, DRIVING_SIGNAL, PRIORITY
FROM PDM_OEE_DB.GOLD.ALERT_QUEUE WHERE STATUS = 'AUTO_ELIGIBLE' ORDER BY PRIORITY ASC;

In [ ]:
-- Fleet OEE overview by wind farm
SELECT WIND_FARM,
    ROUND(AVG(AVAILABILITY), 3) AS AVG_AVAIL,
    ROUND(AVG(PERFORMANCE), 3) AS AVG_PERF,
    ROUND(AVG(QUALITY), 3) AS AVG_QUAL,
    ROUND(AVG(OEE), 3) AS AVG_OEE,
    COUNT(DISTINCT ASSET_ID) AS ASSETS
FROM PDM_OEE_DB.GOLD.GOLD_OEE_METRICS GROUP BY WIND_FARM ORDER BY 1;

---
## Operations Runbook

### Resume All Resources
```sql
ALTER WAREHOUSE COMPUTE_WH RESUME;
ALTER COMPUTE POOL SYSTEM_COMPUTE_POOL_CPU RESUME;
-- Then resume all 8 Dynamic Tables (see docs/DEMO_SCRIPT.md for full list)
```

### Suspend All Resources
```sql
ALTER WAREHOUSE COMPUTE_WH SUSPEND;
ALTER COMPUTE POOL SYSTEM_COMPUTE_POOL_CPU STOP ALL;
ALTER COMPUTE POOL SYSTEM_COMPUTE_POOL_CPU SUSPEND;
-- Suspend all 8 DTs first
```

### Clean Demo Artifacts
```sql
DELETE FROM PDM_OEE_DB.RAW.ERP_WORKORDER WHERE WO_ID LIKE 'WO-AGT%';
DELETE FROM PDM_OEE_DB.RAW.AGENT_ACTION_LOG;
```